# Data Preprocessing and Normalization

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r'C:\Users\kevin\Desktop\VSCode\ScaleDux\Startup_Scoring_Dataset.csv')
df.head()

In [ ]:
df.drop('startup_id', axis=1, inplace=True)

In [ ]:
df['monthly_burn_rate_inr'] = df['monthly_burn_rate_inr'].max() - df['monthly_burn_rate_inr']

In [ ]:
num_cols = df.select_dtypes(include=['int64']).columns

In [ ]:
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# Feature Engineering

In [ ]:
# Defining weights for each feature
weights = {
    'team_experience': 15,
    'market_size_million_usd': 20,
    'monthly_active_users': 25,
    'monthly_burn_rate_inr': 10,
    'funds_raised_inr': 15,
    'valuation_inr': 15
}

In [ ]:
features = list(weights.keys())

In [ ]:
# Calculate composite score by multiplying each feature by its weight and summing them up
df['composite_score'] = sum(df[col] * weight for col, weight in weights.items())

In [ ]:
X = df.drop('composite_score', axis=1).values
y = df['composite_score'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training and Evaluation

In [ ]:
# Build the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(len(features),)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='mse')

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train, 
    validation_data = (X_test, y_test), 
    epochs=150, 
    verbose=1, 
    validation_split=0.2
    )

In [ ]:
# Evaluate the model
preds = model.predict(X_test).flatten()
r2 = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mse)

print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")

In [ ]:
# Plotting training and validation loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Calculate feature importance
input_weights = model.layers[0].get_weights()[0]
feature_importance = np.mean(np.abs(input_weights), axis=1)

In [ ]:
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

In [ ]:
print(importance_df)

In [ ]:
# Plotting feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='coolwarm')
plt.title('Feature Importances (TF Neural Net Input Weights)')
plt.xlabel('Mean Absolute Weight')
plt.tight_layout()
plt.savefig(r'C:\Users\kevin\Desktop\VSCode\ScaleDux\output\feature_importances_tensorflow.png')
plt.show()